# Actividad 1 — Integración SQL–Python y conjunto de trabajo

**Variante 5: Rendimiento comercial por tienda y empleado.**

**Pregunta de negocio:** ¿Cómo se reparte el negocio entre las tiendas y entre quienes atienden? ¿Las diferencias son de volumen o de comportamiento de compra?

Este notebook distingue el **conjunto de trabajo optimizado** (puntos 1.1 y 1.2) de la **comparación sobre pagos individuales** (punto 1.4).


## Preparación — Bibliotecas y conexión a SQL Server


In [1]:
# ACTIVIDAD 1 ANÁLISIS COMERCIAL DE SAKILA
#  importar las bibliotecas necesarias

import pandas as pd
import mssql_python
import os
from dotenv import load_dotenv

print("Pandas:", pd.__version__)
print("Bibliotecas importadas correctamente")
from esquema import nombre_tabla_sql, nombre_columna_sql


Pandas: 3.0.6
Bibliotecas importadas correctamente


In [2]:
# ACTIVIDAD 1 CONEXIÓN A SQL SERVER
# Objetivo: conectarnos a Sakila_Espanol utilizando
# las credenciales guardadas en el archivo .env.

# Cargamos las variables de entorno
load_dotenv()

servidor = os.getenv("DB_SERVER")
puerto = os.getenv("DB_PORT")
base_datos = os.getenv("DB_NAME")
usuario = os.getenv("DB_USER")
contrasena = os.getenv("DB_PASSWORD")

# Establecemos la conexión con SQL Server
conexion = mssql_python.connect(
    f"SERVER={servidor},{puerto};"
    f"DATABASE={base_datos};"
    f"UID={usuario};"
    f"PWD={contrasena};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)

# Verificamos a qué base de datos estamos conectados
cursor = conexion.cursor()
cursor.execute("SELECT DB_NAME()")

resultado = cursor.fetchone()

print("¡Conexión exitosa!")
print("Base de datos:", resultado[0])

cursor.close()

# Mantenemos la conexión abierta para las siguientes consultas.

¡Conexión exitosa!
Base de datos: Sakila_Espanol


## Punto 1.1 — Consulta T-SQL y conjunto de trabajo optimizado

SQL Server realiza las uniones, las agregaciones y los cálculos de participación mediante funciones de ventana. La consulta devuelve una fila por empleado.


In [3]:
# ACTIVIDAD 1  PUNTO 1.1
# Conjunto de trabajo optimizado en SQL Server.
# Objetivo:
# Calcular el rendimiento comercial por tienda y empleado
# directamente en SQL Server, evitando trasladar todos los pagos.

consulta_rendimiento = f"""
WITH rendimiento AS (
    SELECT
        e.{nombre_columna_sql('staff', 'store_id')} AS tienda_id,
        e.{nombre_columna_sql('staff', 'staff_id')} AS empleado_id,
        e.{nombre_columna_sql('staff', 'first_name')} AS nombre,
        e.{nombre_columna_sql('staff', 'last_name')} AS apellido,
        COUNT(p.{nombre_columna_sql('payment', 'payment_id')}) AS cantidad_pagos,
        SUM(p.{nombre_columna_sql('payment', 'amount')}) AS ingresos_totales,
        AVG(p.{nombre_columna_sql('payment', 'amount')}) AS ticket_promedio
    FROM {nombre_tabla_sql('staff')} AS e
    LEFT JOIN {nombre_tabla_sql('payment')} AS p
        ON e.{nombre_columna_sql('staff', 'staff_id')}
         = p.{nombre_columna_sql('payment', 'staff_id')}
    GROUP BY
        e.{nombre_columna_sql('staff', 'store_id')},
        e.{nombre_columna_sql('staff', 'staff_id')},
        e.{nombre_columna_sql('staff', 'first_name')},
        e.{nombre_columna_sql('staff', 'last_name')}
)
SELECT
    tienda_id, empleado_id, nombre, apellido,
    cantidad_pagos, ingresos_totales, ticket_promedio,
    SUM(ingresos_totales) OVER (PARTITION BY tienda_id) AS total_tienda,
    100.0 * ingresos_totales /
        NULLIF(SUM(ingresos_totales) OVER (PARTITION BY tienda_id), 0)
        AS participacion_empleado,
    100.0 * SUM(ingresos_totales) OVER (PARTITION BY tienda_id) /
        NULLIF(SUM(ingresos_totales) OVER (), 0)
        AS participacion_negocio
FROM rendimiento
ORDER BY tienda_id, empleado_id
"""



## Punto 1.2 — Carga del resultado con `pandas.read_sql()`

Ejecutamos la consulta del punto 1.1 y cargamos el conjunto de trabajo optimizado en pandas. Esta consulta no utiliza valores variables, por eso no requiere `params`. Si se agregan filtros variables, deben enviarse mediante `params`, nunca con concatenación ni f-strings.


In [4]:
# Ejecutamos la consulta en SQL Server.
# pandas recibe únicamente los resultados agrupados.
df_rendimiento_optimizado = pd.read_sql(
    consulta_rendimiento,
    conexion
)

# Mostramos los resultados.
display(df_rendimiento_optimizado)

# Verificamos cuántas filas recibimos.
print("Cantidad de filas:", len(df_rendimiento_optimizado))

/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_1505/2398989944.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_rendimiento_optimizado = pd.read_sql(


,tienda_id,empleado_id,nombre,apellido,cantidad_pagos,ingresos_totales,ticket_promedio,total_tienda,participacion_empleado,participacion_negocio
0,1,1,Mike,Hillyer,8054,33482.50,4.157251,33482.50,100.0,49.672465
1,2,2,Jon,Stephens,7990,33924.06,4.245814,33924.06,100.0,50.327534


Cantidad de filas: 2


### Punto 1.3 — Justificación del reparto de trabajo entre SQL Server y pandas

**SQL Server:** se encarga de realizar las operaciones más pesadas del procesamiento de datos. Para nuestro análisis de rendimiento comercial, une las tablas `empleado` y `pago`, cuenta las operaciones, suma los ingresos, calcula el ticket promedio y obtiene las participaciones porcentuales por empleado y tienda mediante funciones de ventana como `SUM(...) OVER (...)`.

De esta manera, las agregaciones se realizan directamente en el motor de base de datos, evitando transferir los 16.044 pagos individuales cuando solamente necesitamos los resultados agrupados.

**pandas:** recibe mediante `pd.read_sql()` el conjunto de trabajo procesado por SQL Server y lo almacena en un DataFrame para facilitar su inspección, análisis y visualización.

Además, en el punto 1.4 utilizamos los datos crudos de los pagos para calcular las mismas métricas con pandas mediante `groupby()` y `transform()`, y comparar los resultados con los obtenidos en SQL Server.

**Justificación de la división del trabajo:**

Elegimos realizar las uniones y agregaciones principales en SQL Server porque está diseñado para procesar grandes volúmenes de datos de manera eficiente. Pandas se utiliza para analizar, validar y visualizar los resultados.

Esta división reduce la cantidad de información transferida desde la base de datos hacia Python y permite comparar ambos enfoques para verificar la consistencia de los cálculos.

## Punto 1.4 — Comparación de SQL Server y pandas con datos crudos

Para este punto se cargan los pagos individuales necesarios para calcular en pandas las mismas métricas que se calculan en T-SQL. 


In [5]:
# ACTIVIDAD 1 CARGA DE DATOS CON PANDAS
# Objetivo: obtener los pagos y la tienda de cada empleado.

consulta_pagos = f"""
SELECT
    p.{nombre_columna_sql('payment', 'payment_id')} AS pago_id,
    p.{nombre_columna_sql('payment', 'staff_id')} AS empleado_id,
    p.{nombre_columna_sql('payment', 'customer_id')} AS cliente_id,
    p.{nombre_columna_sql('payment', 'amount')} AS monto,
    e.{nombre_columna_sql('staff', 'store_id')} AS tienda_id
FROM {nombre_tabla_sql('payment')} AS p
LEFT JOIN {nombre_tabla_sql('staff')} AS e
    ON p.{nombre_columna_sql('payment', 'staff_id')}
     = e.{nombre_columna_sql('staff', 'staff_id')}
"""

# Ejecutamos la consulta SQL
cursor = conexion.cursor()
cursor.execute(consulta_pagos)

# Recuperamos las columnas y los registros
columnas = [columna[0] for columna in cursor.description]
filas = cursor.fetchall()

# Convertimos los resultados en un DataFrame
df_pagos = pd.DataFrame.from_records(filas, columns=columnas)

cursor.close()

print("Cantidad de pagos:", len(df_pagos))
display(df_pagos.head())

Cantidad de pagos: 16044


,pago_id,empleado_id,cliente_id,monto,tienda_id
0,1,1,1,2.99,1
1,2,1,1,0.99,1
2,3,1,1,5.99,1
3,4,2,1,0.99,2
4,5,2,1,9.99,2


In [6]:
# ACTIVIDAD 1  LECTURA DE DATOS CON PANDAS
# Objetivo: cargar los pagos utilizando pd.read_sql().

df_pagos_sql = pd.read_sql(
    consulta_pagos,
    conexion
)

# Verificamos la cantidad de registros obtenidos
print("Cantidad de pagos:", len(df_pagos_sql))

# Mostramos las primeras cinco filas
display(df_pagos_sql.head())

Cantidad de pagos: 16044


/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_1505/4201456945.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pagos_sql = pd.read_sql(


,pago_id,empleado_id,cliente_id,monto,tienda_id
0,1,1,1,2.99,1
1,2,1,1,0.99,1
2,3,1,1,5.99,1
3,4,2,1,0.99,2
4,5,2,1,9.99,2


In [7]:
# ACTIVIDAD 1 RENDIMIENTO POR EMPLEADO
# Objetivo: calcular pagos, ingresos y ticket promedio.

rendimiento_empleados = (
    df_pagos_sql
    .groupby(["tienda_id", "empleado_id"])
    .agg(
        cantidad_pagos=("pago_id", "count"),
        ingresos_totales=("monto", "sum"),
        ticket_promedio=("monto", "mean")
    )
    .reset_index()
)

display(rendimiento_empleados)

,tienda_id,empleado_id,cantidad_pagos,ingresos_totales,ticket_promedio
0,1,1,8054,33482.50,4.157251
1,2,2,7990,33924.06,4.245815


In [8]:
# ACTIVIDAD 1 PARTICIPACIÓN POR EMPLEADO
# Objetivo: calcular qué porcentaje de los ingresos
# de cada tienda corresponde a cada empleado.

rendimiento_empleados["total_tienda"] = (
    rendimiento_empleados
    .groupby("tienda_id")["ingresos_totales"]
    .transform("sum")
)

rendimiento_empleados["participacion_empleado"] = (
    rendimiento_empleados["ingresos_totales"]
    / rendimiento_empleados["total_tienda"]
    * 100
)

display(rendimiento_empleados)

,tienda_id,empleado_id,cantidad_pagos,ingresos_totales,ticket_promedio,total_tienda,participacion_empleado
0,1,1,8054,33482.50,4.157251,33482.50,100.0
1,2,2,7990,33924.06,4.245815,33924.06,100.0


In [9]:
# ACTIVIDAD 1 PARTICIPACIÓN DE CADA TIENDA
# Objetivo: calcular qué porcentaje del ingreso total
# del negocio corresponde a cada tienda.

# Sumamos los ingresos de todas las tiendas.
ingresos_negocio = rendimiento_empleados["ingresos_totales"].sum()

# Calculamos el porcentaje que aporta cada tienda.
rendimiento_empleados["participacion_negocio"] = (
    rendimiento_empleados["total_tienda"]
    / ingresos_negocio
    * 100
)

print("Ingresos totales del negocio:", round(ingresos_negocio, 2))

display(
    rendimiento_empleados[
        ["tienda_id", "total_tienda", "participacion_negocio"]
    ]
)

Ingresos totales del negocio: 67406.56


,tienda_id,total_tienda,participacion_negocio
0,1,33482.50,49.672465
1,2,33924.06,50.327535


In [10]:
# ACTIVIDAD 1 VALIDACIÓN SQL VS PANDAS
# Objetivo: comprobar que ambos métodos
# calculan los mismos ingresos por empleado.

consulta_validacion = f"""
SELECT
    e.{nombre_columna_sql('staff', 'store_id')} AS tienda_id,
    p.{nombre_columna_sql('payment', 'staff_id')} AS empleado_id,
    COUNT(*) AS cantidad_pagos,
    SUM(p.{nombre_columna_sql('payment', 'amount')}) AS ingresos_sql
FROM {nombre_tabla_sql('payment')} AS p
INNER JOIN {nombre_tabla_sql('staff')} AS e
    ON p.{nombre_columna_sql('payment', 'staff_id')}
     = e.{nombre_columna_sql('staff', 'staff_id')}
GROUP BY
    e.{nombre_columna_sql('staff', 'store_id')},
    p.{nombre_columna_sql('payment', 'staff_id')}
ORDER BY tienda_id, empleado_id
"""

# Ejecutamos el cálculo en SQL Server.
df_sql = pd.read_sql(consulta_validacion, conexion)

# Comparamos con los resultados calculados en pandas.
comparacion = df_sql.merge(
    rendimiento_empleados[
        ["tienda_id", "empleado_id", "cantidad_pagos", "ingresos_totales"]
    ],
    on=["tienda_id", "empleado_id"],
    suffixes=("_sql", "_pandas")
)

# Calculamos la diferencia entre ambos resultados.
comparacion["diferencia_ingresos"] = (
    comparacion["ingresos_sql"]
    - comparacion["ingresos_totales"]
).round(2)

display(comparacion)

/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_1505/2495346852.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql = pd.read_sql(consulta_validacion, conexion)


,tienda_id,empleado_id,cantidad_pagos_sql,ingresos_sql,cantidad_pagos_pandas,ingresos_totales,diferencia_ingresos
0,1,1,8054,33482.50,8054,33482.50,0.0
1,2,2,7990,33924.06,7990,33924.06,-0.0


In [11]:
# ACTIVIDAD 1 COMPARACIÓN DE RENDIMIENTO
# Objetivo: comparar el tiempo de cálculo
# de ingresos por empleado en SQL y pandas.

from time import perf_counter

# Medimos cuánto tarda SQL Server en calcular los resultados.
inicio_sql = perf_counter()

resultado_sql = pd.read_sql(consulta_validacion, conexion)

fin_sql = perf_counter()

# Medimos cuánto tarda pandas en calcular los mismos resultados.
inicio_pandas = perf_counter()

resultado_pandas = (
    df_pagos_sql
    .groupby(["tienda_id", "empleado_id"])
    .agg(
        cantidad_pagos=("pago_id", "count"),
        ingresos_totales=("monto", "sum")
    )
    .reset_index()
)

fin_pandas = perf_counter()

# Calculamos los tiempos transcurridos.
tiempo_sql = fin_sql - inicio_sql
tiempo_pandas = fin_pandas - inicio_pandas

print(f"Tiempo SQL Server: {tiempo_sql:.6f} segundos")
print(f"Tiempo pandas: {tiempo_pandas:.6f} segundos")

Tiempo SQL Server: 0.023016 segundos
Tiempo pandas: 0.002028 segundos


/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_1505/2091895298.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  resultado_sql = pd.read_sql(consulta_validacion, conexion)


 Comparación de resultados y tiempos de ejecución

Se analizaron 16.044 pagos de la base de datos Sakila_Espanol.

Los ingresos y la cantidad de operaciones calculados mediante SQL Server coincidieron con los obtenidos utilizando pandas.

Resultados comerciales:
- Tienda 1: 8.054 pagos, ingresos de 33.482,50 y ticket promedio de 4,16.
- Tienda 2: 7.990 pagos, ingresos de 33.924,06 y ticket promedio de 4,25.
- Ingresos totales: 67.406,56.

La tienda 2 generó más ingresos a pesar de registrar menos operaciones. Esto indica que la diferencia está relacionada con el ticket promedio y no solamente con el volumen de pagos.

Comparación de rendimiento:
- Tiempo SQL Server: 0.023016 segundos
- Tiempo pandas: 0.002028 segundos

En esta ejecución, pandas presentó un menor tiempo de procesamiento. Sin embargo, la comparación tiene limitaciones: la medición de SQL incluye la consulta y transferencia de resultados, mientras que pandas utiliza datos previamente cargados en memoria.

Por lo tanto, estos tiempos no permiten determinar cuál tecnología es más eficiente en términos generales.

## Controles de calidad de datos


In [12]:
# ACTIVIDAD 1 CONTROL DE CALIDAD DE DATOS
# Objetivo: detectar pagos sin un empleado válido.

# Seleccionamos los pagos que no tienen tienda asociada.
pagos_sin_empleado = df_pagos_sql[
    df_pagos_sql["tienda_id"].isna()
]

print("Cantidad total de pagos:", len(df_pagos_sql))
print("Pagos sin empleado válido:", len(pagos_sin_empleado))

display(pagos_sin_empleado.head())

Cantidad total de pagos: 16044
Pagos sin empleado válido: 0


,pago_id,empleado_id,cliente_id,monto,tienda_id


In [13]:
# ACTIVIDAD 1 CONTROL DE CALIDAD
# Objetivo: detectar empleados sin pagos registrados.

consulta_empleados_sin_pagos = f"""
SELECT
    e.{nombre_columna_sql('staff', 'staff_id')} AS empleado_id,
    e.{nombre_columna_sql('staff', 'first_name')} AS nombre,
    e.{nombre_columna_sql('staff', 'last_name')} AS apellido,
    e.{nombre_columna_sql('staff', 'store_id')} AS tienda_id,
    COUNT(p.{nombre_columna_sql('payment', 'payment_id')}) AS cantidad_pagos
FROM {nombre_tabla_sql('staff')} AS e
LEFT JOIN {nombre_tabla_sql('payment')} AS p
    ON e.{nombre_columna_sql('staff', 'staff_id')}
     = p.{nombre_columna_sql('payment', 'staff_id')}
GROUP BY
    e.{nombre_columna_sql('staff', 'staff_id')},
    e.{nombre_columna_sql('staff', 'first_name')},
    e.{nombre_columna_sql('staff', 'last_name')},
    e.{nombre_columna_sql('staff', 'store_id')}
HAVING COUNT(p.{nombre_columna_sql('payment', 'payment_id')}) = 0
"""

df_empleados_sin_pagos = pd.read_sql(
    consulta_empleados_sin_pagos,
    conexion
)

print("Empleados sin pagos:", len(df_empleados_sin_pagos))

display(df_empleados_sin_pagos)

/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_1505/3254800069.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_empleados_sin_pagos = pd.read_sql(


Empleados sin pagos: 0


,empleado_id,nombre,apellido,tienda_id,cantidad_pagos


In [14]:
# ACTIVIDAD 1 CONTROL DE CALIDAD
# Objetivo: identificar clientes cuyos pagos
# fueron procesados en más de una tienda.

consulta_clientes_ambas_tiendas = f"""
SELECT
    p.{nombre_columna_sql('payment', 'customer_id')} AS cliente_id,
    COUNT(DISTINCT e.{nombre_columna_sql('staff', 'store_id')}) AS cantidad_tiendas
FROM {nombre_tabla_sql('payment')} AS p
INNER JOIN {nombre_tabla_sql('staff')} AS e
    ON p.{nombre_columna_sql('payment', 'staff_id')}
     = e.{nombre_columna_sql('staff', 'staff_id')}
GROUP BY p.{nombre_columna_sql('payment', 'customer_id')}
HAVING COUNT(DISTINCT e.{nombre_columna_sql('staff', 'store_id')}) > 1
"""

df_clientes_ambas_tiendas = pd.read_sql(
    consulta_clientes_ambas_tiendas,
    conexion
)

print(
    "Clientes con pagos procesados en ambas tiendas:",
    len(df_clientes_ambas_tiendas)
)

display(df_clientes_ambas_tiendas.head())

Clientes con pagos procesados en ambas tiendas: 599


/var/folders/cn/rlqrdxlx7_s99d2kjb3ztf1c0000gn/T/ipykernel_1505/3300478183.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_clientes_ambas_tiendas = pd.read_sql(


,cliente_id,cantidad_tiendas
0,1,2
1,2,2
2,3,2
3,4,2
4,5,2


Controles previos de integridad
Se verificó la consistencia de las relaciones utilizadas para construir el conjunto de trabajo. No se detectaron pagos sin empleado válido ni empleados sin operaciones.
El análisis detallado de calidad de datos, incluida la identificación de 599 clientes con pagos procesados en ambas tiendas, se encuentra documentado en la Actividad 2.

 Conclusión general Actividad 1

Se logró integrar SQL Server con Python mediante 'pandas.read_sql()', obteniendo los datos necesarios para analizar los ingresos por empleado y tienda.

Se comparó el cálculo de los ingresos totales mediante dos enfoques: una consulta T-SQL con agregaciones realizadas en el motor de base de datos y un procesamiento con pandas a partir de los pagos individuales.

Validación de resultados: Ambos enfoques obtuvieron los mismos ingresos por empleado, con diferencias nulas dentro de la precisión numérica utilizada.

Distribución del procesamiento: SQL Server resulta apropiado para ejecutar uniones, filtros y agregaciones antes de transferir los resultados a Python, reduciendo la cantidad de datos transportados. Pandas permite realizar transformaciones, verificaciones y análisis adicionales de manera flexible.

Comparación de rendimiento: Se midieron los tiempos de ejecución de ambos enfoques mediante 'perf_counter()'. Estos tiempos permiten comparar el comportamiento observado en el entorno de trabajo, aunque una única medición no es suficiente para establecer cuál es más rápido en general.

En conclusión, ambos métodos permiten obtener resultados consistentes. Para construir el conjunto de trabajo se prioriza el procesamiento en SQL Server, mientras que pandas se utiliza para el análisis y la validación de los datos.

In [15]:
import sys
print(sys.executable)

/Users/dalmi/Desktop/TP2-Sakila-Analisis/.venv/bin/python


In [16]:
import os

print("DYLD_LIBRARY_PATH:", os.getenv("DYLD_LIBRARY_PATH"))
print("DYLD_FALLBACK_LIBRARY_PATH:", os.getenv("DYLD_FALLBACK_LIBRARY_PATH"))

DYLD_LIBRARY_PATH: /Library/Frameworks/Python.framework/Versions/3.12/lib
DYLD_FALLBACK_LIBRARY_PATH: None
